# SPLADE: бенчмарк на NFCorpus (zero-shot, быстрый eval)

Копия `splade_benchmark_scifact.ipynb` для другого набора. **Единственное
отличие — имя набора** в клетке 2 (`DATASET = "nfcorpus"`); весь остальной код
идентичен. Это иллюстрирует модульность: новый набор = новый блокнот-копия с
одной изменённой строкой, общий код в `splade_lab/` не трогаем.

NFCorpus (BEIR) — медицинский IR-набор, ~3.6K документов и ~323 запроса. Как и
SciFact, индексируется за секунды (против ~1.5 ч на полном MS MARCO). Главная
метрика — **nDCG@10**, оценка zero-shot моделей, обученных на MS MARCO.

## Порядок
1. Setup. 2. Конфиг набора. 3. Данные. 4. Модели. 5. Бенчмарк.
6. Сводная таблица. 7. Значимость. 8. Обоснование (см. scifact-блокнот).

## 1. Setup

In [1]:
# Первый запуск на новой машине (в активированном venv) — раскомментировать:
# %pip install -r requirements.txt
%load_ext autoreload
%autoreload 2

import json
import torch

from splade_lab.train import pick_device

device = pick_device()
print(f"torch {torch.__version__} | устройство: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

torch 2.5.1+cu124 | устройство: cuda (NVIDIA A100 80GB PCIe)


## 2. Конфиг набора (NFCorpus)

**Вот и всё отличие от scifact-блокнота:** имя набора. Остальное — без изменений.

In [2]:
from splade_lab import datasets

DATASET = "nfcorpus"  # <-- единственное отличие от scifact-блокнота
DATA = datasets.beir_config(DATASET, num_eval_queries=-1)

print(f"Набор: {DATA['name']} | split={DATA['split']}")
print(f"URL:   {DATA['url']}")
print(f"Каталог на диске: {datasets.dataset_dir(DATA)}")
print(f"Известные наборы: {sorted(datasets.BEIR_DATASETS)}")

Набор: nfcorpus | split=test
URL:   https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip
Каталог на диске: /home/uvuv/workspace/sandbox_03/data/beir/nfcorpus
Известные наборы: ['arguana', 'fiqa', 'nfcorpus', 'scidocs', 'scifact', 'trec-covid', 'webis-touche2020']


## 3. Данные (скачиваются один раз)

In [3]:
ds_dir = datasets.prepare_beir(DATA)

print(f"\nКаталог: {ds_dir}")
print("Строк в файлах:", datasets.dataset_stats(ds_dir))

with open(ds_dir / "queries.tsv", encoding="utf-8") as f:
    print("\nПример запроса:  ", f.readline().strip()[:120])
with open(ds_dir / "collection.tsv", encoding="utf-8") as f:
    print("Пример документа:", f.readline().strip()[:120], "...")

[beir] nfcorpus уже готов: /home/uvuv/workspace/sandbox_03/data/beir/nfcorpus

Каталог: /home/uvuv/workspace/sandbox_03/data/beir/nfcorpus
Строк в файлах: {'collection.tsv': 3633, 'queries.tsv': 323, 'qrels.tsv': 12334}

Пример запроса:   PLAIN-2	Do Cholesterol Statin Drugs Cause Breast Cancer?
Пример документа: MED-10	Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that  ...


## 4. Какие модели сравниваем

In [4]:
from splade_lab import modeling

MODELS = [
    {"version": "v1_splade_max", "label": "v1_splade_max"},
    {"version": "v2_splade_doc", "label": "v2_splade_doc"},
]

for m in MODELS:
    try:
        run = modeling.resolve_model_run(version=m.get("version"),
                                         run_id=m.get("run_id"),
                                         run_dir=m.get("run_dir"))
        print(f"{m['label']:>16} -> {run}")
    except FileNotFoundError as e:
        print(f"{m['label']:>16} -> [нет модели] {e}")

   v1_splade_max -> /home/uvuv/workspace/sandbox_03/outputs/v1_splade_max/20260613-005141
   v2_splade_doc -> /home/uvuv/workspace/sandbox_03/outputs/v2_splade_doc/20260613-180612


## 5. Бенчмарк (zero-shot прогон по NFCorpus)

In [5]:
from splade_lab import benchmark

RECALL_KS = (10, 100)
NDCG_KS = (10,)

bench_results = {}
for m in MODELS:
    try:
        res = benchmark.run_benchmark(
            version=m.get("version"), run_id=m.get("run_id"),
            run_dir=m.get("run_dir"), query_encoder=m.get("query_encoder"),
            data_cfg=DATA, device=device,
            recall_ks=RECALL_KS, ndcg_ks=NDCG_KS,
            label=m["label"], save=True)
        bench_results[m["label"]] = res
    except (FileNotFoundError, RuntimeError) as e:
        print(f"[skip] {m['label']}: {e}")

for label, res in bench_results.items():
    agg = res["aggregate"]
    print(f"{label:>16}: nDCG@10={agg.get('ndcg@10'):.4f} "
          f"recall@10={agg.get('recall@10'):.4f} "
          f"| index={res['timing']['index_s']}s search={res['timing']['search_s']}s")

[bench] v1_splade_max (src=20260613-005141) на nfcorpus qe=mlm device=cuda
[progress] лог прогресса: /home/uvuv/workspace/sandbox_03/outputs/logs/progress_20260614-225721.log


/home/uvuv/workspace/sandbox_03/splade_lab/evaluate.py:73: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  D = torch.sparse_csr_tensor(


[bench] nfcorpus | mrr@10=0.5224 recall@10=0.1545 recall@100=0.2733 ndcg@10=0.3185 avg_nnz_doc=339.4429 avg_nnz_query=20.5480 | index=3.13s search=0.2s
[bench] артефакты: /home/uvuv/workspace/sandbox_03/outputs/benchmark/nfcorpus/v1_splade_max__20260613-005141
[bench] v2_splade_doc (src=20260613-180612) на nfcorpus qe=bow device=cuda
[bench] nfcorpus | mrr@10=0.5339 recall@10=0.1457 recall@100=0.2499 ndcg@10=0.3103 avg_nnz_doc=249.9659 avg_nnz_query=4.9257 | index=2.81s search=0.12s
[bench] артефакты: /home/uvuv/workspace/sandbox_03/outputs/benchmark/nfcorpus/v2_splade_doc__20260613-180612
   v1_splade_max: nDCG@10=0.3185 recall@10=0.1545 | index=3.13s search=0.2s
   v2_splade_doc: nDCG@10=0.3103 recall@10=0.1457 | index=2.81s search=0.12s


## 6. Сводная таблица метрик на NFCorpus

In [6]:
import pandas as pd

rows = []
for label, res in bench_results.items():
    agg = res["aggregate"]
    rows.append({
        "label": label,
        "version": res["version"],
        "run_id": res["run_id"],
        "query_encoder": res["query_encoder"],
        "ndcg@10": agg.get("ndcg@10"),
        "recall@10": agg.get("recall@10"),
        "recall@100": agg.get("recall@100"),
        "mrr@10": agg.get("mrr@10"),
        "avg_nnz_doc": agg.get("avg_nnz_doc"),
        "avg_nnz_query": agg.get("avg_nnz_query"),
        "n_eval_queries": agg.get("n_eval_queries"),
        "n_corpus_docs": agg.get("n_corpus_docs"),
        "index_s": res["timing"]["index_s"],
        "search_s": res["timing"]["search_s"],
    })

df = pd.DataFrame(rows).sort_values("ndcg@10", ascending=False).reset_index(drop=True)
out_csv = benchmark.BENCH_OUTPUTS / DATASET / "comparison.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print(df.to_string(index=False))
print(f"\n[bench] CSV: {out_csv}")
df

        label       version          run_id query_encoder  ndcg@10  recall@10  recall@100   mrr@10  avg_nnz_doc  avg_nnz_query  n_eval_queries  n_corpus_docs  index_s  search_s
v1_splade_max v1_splade_max 20260613-005141           mlm 0.318496   0.154474    0.273306 0.522426   339.442885      20.547988             323           3633     3.13      0.20
v2_splade_doc v2_splade_doc 20260613-180612           bow 0.310346   0.145736    0.249886 0.533895   249.965868       4.925697             323           3633     2.81      0.12

[bench] CSV: /home/uvuv/workspace/sandbox_03/outputs/benchmark/nfcorpus/comparison.csv


,label,version,run_id,query_encoder,ndcg@10,recall@10,recall@100,mrr@10,avg_nnz_doc,avg_nnz_query,n_eval_queries,n_corpus_docs,index_s,search_s
0,v1_splade_max,v1_splade_max,20260613-005141,mlm,0.318496,0.154474,0.273306,0.522426,339.442885,20.547988,323,3633,3.13,0.20
1,v2_splade_doc,v2_splade_doc,20260613-180612,bow,0.310346,0.145736,0.249886,0.533895,249.965868,4.925697,323,3633,2.81,0.12


## 7. Статистическая значимость различий

Парные тесты по nDCG@10 (bootstrap + t-тест + Wilcoxon) на одних и тех же
запросах NFCorpus. Обоснование корректности — в разделе 8 scifact-блокнота.

In [7]:
from splade_lab import stats

METRIC = "ndcg@10"
ALPHA = 0.05

labels = list(bench_results)
if len(labels) < 2:
    print("Нужно >=2 модели в bench_results для сравнения значимости.")
else:
    baseline = labels[0]
    pq_base = bench_results[baseline]["per_query"]
    pvals = {}
    for label in labels[1:]:
        cmp = stats.compare_pair(pq_base, bench_results[label]["per_query"],
                                 metric=METRIC, name_a=baseline, name_b=label,
                                 alpha=ALPHA, n_boot=10000, seed=0)
        pvals[f"{baseline}_vs_{label}"] = cmp["p_bootstrap"]
        print(f"\n=== {baseline}  vs  {label}  ({METRIC}) ===")
        print(f"  mean({baseline}) = {cmp['mean_a']:.4f} | mean({label}) = {cmp['mean_b']:.4f}")
        print(f"  Δ = {cmp['delta']:+.4f}  ({cmp['rel_improvement_pct']:+.1f}%)  "
              f"95% ДИ [{cmp['ci95_low']:+.4f}, {cmp['ci95_high']:+.4f}]")
        print(f"  p(bootstrap) = {cmp['p_bootstrap']:.4g} | "
              f"p(t-test) = {cmp['p_ttest']:.4g} | p(Wilcoxon) = {cmp['p_wilcoxon']:.4g}")
        print(f"  n_queries = {cmp['n_queries']} | значимо (bootstrap, α={ALPHA}): {cmp['significant']}")

    if len(pvals) > 1:
        print("\n=== Поправка Холма–Бонферрони ===")
        for key, info in stats.holm_correction(pvals, alpha=ALPHA).items():
            print(f"  {key}: p={info['p']:.4g} -> p_adj={info['p_adj']:.4g} "
                  f"| отвергаем H0: {info['reject']}")


=== v1_splade_max  vs  v2_splade_doc  (ndcg@10) ===
  mean(v1_splade_max) = 0.3185 | mean(v2_splade_doc) = 0.3103
  Δ = -0.0081  (-2.6%)  95% ДИ [-0.0204, +0.0042]
  p(bootstrap) = 0.1874 | p(t-test) = 0.19 | p(Wilcoxon) = 0.1119
  n_queries = 323 | значимо (bootstrap, α=0.05): False


## 8. Обоснование

См. раздел 8 в `splade_benchmark_scifact.ipynb` — методология одинакова. Главное:
согласованный значимый прирост **на нескольких наборах** (SciFact + NFCorpus +
...) — более сильное свидетельство улучшения модели, чем на одном наборе.